1. Doc2Vec를 이용해서 ratings_train 데이터를 학습 
2. 학습된 모델을 저장 
3. 모델을 로드 하고 모델을 이용하여 ratings_test 데이터를 벡터화 
    - 데이터를 token화 
    - infer_vector() 함수를 이용하여 벡터화
4. LSTM 모델을 이용해서 학습, 검증 

In [2]:
import pandas as pd 
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import numpy as np 
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

In [3]:
# 모델 학습 저장 
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df = df[:100]
komoran = Komoran()
def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']
    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

In [4]:
tokenized_sentence = [ tokenize(text) for text in df['document'] ]

In [6]:
tagged_data = [
    TaggedDocument(words = doc, tags = [str(i)]) for i, doc in enumerate(tokenized_sentence)
]

In [7]:
d2v = Doc2Vec(
    tagged_data, vector_size=64, window = 5, min_count =1, workers = 2, epochs = 20
)

In [ ]:
# 학습된 모델을 저장
d2v.save('my_model.model')

In [9]:
# 학습된 모델을 로드 
loaded_doc2vec = Doc2Vec.load('my_doc2vec.model')

In [10]:
# 평가용 데이터셋 test 데이터 로드 
df = pd.read_csv("../data/ratings_test.txt", sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        50000 non-null  int64
 1   document  49997 non-null  str  
 2   label     50000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB


In [11]:
df.dropna(inplace=True)
df.drop_duplicates('document', inplace=True)

In [12]:
df = df[:5000]

In [28]:
df['label'].value_counts()

label
1    2519
0    2481
Name: count, dtype: int64

In [ ]:
# 토큰화 함수는 DOc2Vec에서 사용한 토큰화 함수와 같은 함수를 이용

tokenized_sentences = [ tokenize(text) for text in df['document'] ]

In [29]:
tokenized_sentences

[['굳'],
 ['GDNTOPCLASSINTHECLUB'],
 ['평점', '나쁘', '더더욱'],
 ['완전', '막', '장임', '돈', '주', '보'],
 ['3D', '텐', '왜', '3D', '나오', '심기', '불편'],
 ['음악', '주가', '되', '최고', '음악', '영화'],
 ['쓰레기'],
 ['마치', '미국', '애니', '튀어나오', '창의력', '없', '로봇', '디자인', '고개', '젖'],
 ['갈수록',
  '개판',
  '중국',
  '영화',
  '내용',
  '없',
  '폼',
  '잡',
  '끝나',
  '말도',
  '안',
  '무기',
  '유치',
  '한',
  'cg',
  '남무',
  '그립',
  '동사서독',
  '같',
  '영화',
  '이건',
  '류',
  '아',
  '류',
  '작'],
 ['이별', '아픔', '뒤', '찾아오', '새롭', '인연', '기쁨', 'But', '사람', '그렇'],
 [],
 ['한국', '독립', '영화', '한계', '그렇게 아버지가 된다', '비교'],
 ['청춘',
  '아름답다',
  '아름답',
  '이성',
  '흔들',
  '찰나',
  '아름답',
  '잘',
  '포착',
  '아름답',
  '수채화',
  '같',
  '퀴어',
  '영화'],
 ['눈', '보이', '반전', '영화', '흡인력', '사라지'],
 ['스토리',
  '연출',
  '연기',
  '비주얼',
  '영화',
  '기본',
  '조차',
  '안',
  '영화',
  '영화',
  '찍',
  '김문',
  '옥',
  '감독',
  '영화',
  '경력',
  'OO',
  '조무래기',
  '영화',
  '평론',
  '같',
  '마인드',
  '빠지'],
 ['소위', '평점'],
 ['최고', '!!!', '!!!', '!!!', '!!!', '!!', '!!'],
 ['발', '연기', '도저히', '못', '보', '진짜'

In [16]:
X_vector = []

for sentence in tokenized_sentences:
    vec = loaded_doc2vec.infer_vector(sentence)
    X_vector.append(vec)

X = np.array(X_vector)
y = df['label'].values

In [30]:
X.shape

(5000, 64)

In [19]:
# Dataset, collate_fn, DataLoader 생성 
class LSTMDataset(Dataset):
    def __init__(self, vectors, labels):
        self.labels = labels
        self.data = vectors
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

In [20]:
dataset = LSTMDataset(X, y)
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [21]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle = True)

In [23]:
class LSTMCLF(nn.Module):
    def __init__(self, input_dim, hidden_size, num_classes, dropout = 0.5, head_type = 'last'):
        super().__init__()

        self.head_type = head_type
        # input_dim : 벡터화 데이터의 차원의 수
        
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True)
        # 과적합 방지용 dropout
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # 차원을 확장 
        x = x.unsqueeze(1)    # [batch_size, 1, input_dim]
        # LSTM 결과 값 A, (B,C)
        lstm_out, (hidden, cell) = self.lstm(x)
        if self.head_type == 'last':
            last_hidden = hidden.squeeze(0)
        elif self.head_type == 'mean':
            # 모든 층의 값들의 평균을 구한다. 
            # lstm_out -> [batch_size, seq_len, hidden_size]
            last_hidden = torch.mean( lstm_out, dim=1 )    # [batch_size, hidden_size]
        elif self.head_type == 'max':
            last_hidden, _ = torch.max(lstm_out, dim = 1)

        dropout_hidden = self.dropout(last_hidden)

        # return self.fc(last_hidden)
        return self.fc(dropout_hidden)

In [26]:
model = LSTMCLF(input_dim=64, hidden_size=128, num_classes=2, 
                head_type='last', dropout = 0.3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [27]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0 
    total_train = 0 

    for inputs, labels in tqdm(train_loader, desc = f"Epoch {epoch+1} / {epochs} "):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    train_acc = (correct_train / total_train) * 100
    avg_train_loss  = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim = 1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)

    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)

    if (epoch + 1) % 10 == 0:
        print(f"LSTM 에폭 결과 : Train Loss {round(avg_train_loss, 4)} Train Acc {train_acc} " )
        print(f"LSTM 에폭 결과 : Vali Loss {round(avg_val_loss, 4)} Vali Acc {val_acc}")

Epoch 10 / 50 : 100%|██████████| 63/63 [00:00<00:00, 220.14it/s]


LSTM 에폭 결과 : Train Loss 0.535 Train Acc 72.7 
LSTM 에폭 결과 : Vali Loss 0.5223 Vali Acc 72.6


Epoch 20 / 50 : 100%|██████████| 63/63 [00:00<00:00, 294.64it/s]


LSTM 에폭 결과 : Train Loss 0.5239 Train Acc 72.95 
LSTM 에폭 결과 : Vali Loss 0.5261 Vali Acc 71.6


Epoch 30 / 50 : 100%|██████████| 63/63 [00:00<00:00, 269.49it/s]


LSTM 에폭 결과 : Train Loss 0.5127 Train Acc 73.775 
LSTM 에폭 결과 : Vali Loss 0.5203 Vali Acc 72.1


Epoch 40 / 50 : 100%|██████████| 63/63 [00:00<00:00, 301.82it/s]


LSTM 에폭 결과 : Train Loss 0.4985 Train Acc 74.7 
LSTM 에폭 결과 : Vali Loss 0.5232 Vali Acc 72.3


Epoch 50 / 50 : 100%|██████████| 63/63 [00:00<00:00, 259.95it/s]

LSTM 에폭 결과 : Train Loss 0.4852 Train Acc 76.075 
LSTM 에폭 결과 : Vali Loss 0.523 Vali Acc 71.39999999999999
